# FastAPI: Building High-Performance ML Model APIs

## What Is FastAPI?

Imagine a restaurant. The kitchen (your ML model) makes the food. But customers can't walk into the kitchen — they need a **waiter** to take their order, bring it to the kitchen, and deliver the food.  
**FastAPI** is that waiter — it takes requests from users/apps, passes them to your model, and returns the predictions.

**FastAPI** is a modern Python web framework for building APIs:
- **Fast**: one of the fastest Python frameworks (on par with Node.js and Go)
- **Async**: handles many requests simultaneously without blocking
- **Auto-docs**: generates interactive API docs at `/docs` automatically
- **Type-safe**: uses Python type hints + Pydantic for automatic validation

## Resources

- **Docs**: [https://fastapi.tiangolo.com/](https://fastapi.tiangolo.com/)
- **GitHub**: [https://github.com/tiangolo/fastapi](https://github.com/tiangolo/fastapi)
- **YouTube — FastAPI tutorial**: [https://www.youtube.com/watch?v=0RS9W8MtZe4](https://www.youtube.com/watch?v=0RS9W8MtZe4)
- **YouTube — FastAPI for ML**: [https://www.youtube.com/watch?v=Osi8PNe0NOQ](https://www.youtube.com/watch?v=Osi8PNe0NOQ)

## Installation

```bash
pip install fastapi uvicorn[standard]
# uvicorn = ASGI server that runs FastAPI apps
# [standard] includes websockets, httptools for better performance
```

**In this notebook**: we use FastAPI's `TestClient` which doesn't need a running server.  
All API calls work directly in the notebook!

In [ ]:
import json, pickle, time
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from typing import List, Optional

try:
    from fastapi import FastAPI, HTTPException, Depends, Header, BackgroundTasks
    from fastapi.middleware.cors import CORSMiddleware
    from fastapi.testclient import TestClient
    from pydantic import BaseModel, Field, validator
    FASTAPI_AVAILABLE = True
    import fastapi
    print(f"FastAPI version: {fastapi.__version__}")
    print("TestClient available — all API calls work in this notebook!")
except ImportError:
    FASTAPI_AVAILABLE = False
    print("FastAPI not installed — showing code structure with simulated output")
    print("Install: pip install fastapi uvicorn[standard]")

# Train a model to serve
np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train_s, y_train)
print(f"Model trained: {model.score(scaler.transform(X_test), y_test):.3f} accuracy")

## Core Concept 1: Pydantic Models — Request/Response Validation

Pydantic models define the **shape** of data coming in and going out.  
FastAPI automatically validates that requests match the schema — no manual checking needed.

In [ ]:
if FASTAPI_AVAILABLE:
    from pydantic import BaseModel, Field, field_validator
    import pydantic

    # ── Request models ────────────────────────────────────────────────────────
    class PredictionRequest(BaseModel):
        """Single prediction request."""
        features: List[float] = Field(
            ..., 
            min_length=10, max_length=10,
            description="List of 10 feature values",
            example=[1.2, -0.5, 0.8, 2.1, -1.3, 0.4, -0.9, 1.7, 0.2, -0.6]
        )
        return_probabilities: bool = Field(
            default=True,
            description="Whether to return class probabilities"
        )

        @field_validator('features')
        @classmethod
        def features_must_be_finite(cls, v):
            if any(not np.isfinite(f) for f in v):
                raise ValueError('Features must be finite numbers (no NaN or Inf)')
            return v

    class BatchPredictionRequest(BaseModel):
        """Batch prediction request."""
        instances: List[List[float]] = Field(
            ..., max_length=1000,
            description="List of feature vectors (max 1000 per request)"
        )

    # ── Response models ───────────────────────────────────────────────────────
    class PredictionResponse(BaseModel):
        prediction:   int
        confidence:   float
        probabilities: Optional[List[float]] = None
        latency_ms:   float
        model_version: str

    class BatchPredictionResponse(BaseModel):
        predictions:   List[int]
        probabilities: Optional[List[List[float]]] = None
        n_instances:   int
        latency_ms:    float

    class HealthResponse(BaseModel):
        status:        str
        model_loaded:  bool
        model_version: str
        uptime_seconds: float

    # Test Pydantic validation
    valid_req = PredictionRequest(features=[1.0]*10, return_probabilities=True)
    print(f"Valid request: {valid_req.model_dump()}")
    print()

    try:
        bad_req = PredictionRequest(features=[1.0]*5)  # wrong length
    except Exception as e:
        print(f"Invalid request caught: {type(e).__name__}")
        print("FastAPI returns HTTP 422 Unprocessable Entity with a detailed error message")

else:
    print("Pydantic model example (simulated):")
    print("""
  from pydantic import BaseModel, Field

  class PredictionRequest(BaseModel):
      features: List[float] = Field(..., min_length=10, max_length=10)
      return_probabilities: bool = True

      @field_validator('features')
      def features_must_be_finite(cls, v):
          if any(not np.isfinite(f) for f in v):
              raise ValueError('Features must be finite')
          return v

  # FastAPI automatically validates incoming JSON against this schema
  # Invalid requests → HTTP 422 with detailed error (no manual try/except needed)
  """)

## Core Concept 2: Building the ML Serving Application

In [ ]:
if FASTAPI_AVAILABLE:
    START_TIME = time.time()
    MODEL_VERSION = "1.0.0"
    prediction_log = []  # simple in-memory log

    app = FastAPI(
        title="ML Model Serving API",
        description="Serves a RandomForest classifier with full validation",
        version=MODEL_VERSION,
        docs_url="/docs",       # Swagger UI at /docs
        redoc_url="/redoc",     # ReDoc UI at /redoc
    )

    # CORS middleware — allow web browsers to call this API
    app.add_middleware(
        CORSMiddleware,
        allow_origins=["*"],    # In production: restrict to specific origins
        allow_methods=["GET", "POST"],
        allow_headers=["*"],
    )

    # ── Dependencies ──────────────────────────────────────────────────────────
    API_KEY = "secret-key-12345"  # In production: use environment variables!

    def verify_api_key(x_api_key: str = Header(default="")):
        """Dependency: verify API key header."""
        if x_api_key != API_KEY:
            raise HTTPException(status_code=401, detail="Invalid or missing API key")
        return x_api_key

    # ── Background task ───────────────────────────────────────────────────────
    def log_prediction(request_data: dict, response_data: dict):
        """Runs AFTER the response is sent — doesn't slow down the API."""
        prediction_log.append({
            "timestamp": time.time(),
            "request": request_data,
            "response": response_data,
        })
        # In production: write to database, send to monitoring system, etc.

    # ── Endpoints ─────────────────────────────────────────────────────────────
    @app.get("/health", response_model=HealthResponse, tags=["Monitoring"])
    def health_check():
        """Health check endpoint — used by Kubernetes liveness probes."""
        return HealthResponse(
            status="healthy",
            model_loaded=True,
            model_version=MODEL_VERSION,
            uptime_seconds=time.time() - START_TIME,
        )

    @app.get("/model-info", tags=["Monitoring"])
    def model_info():
        """Return metadata about the loaded model."""
        return {
            "model_type": type(model).__name__,
            "n_estimators": model.n_estimators,
            "n_features": model.n_features_in_,
            "classes": model.classes_.tolist(),
            "version": MODEL_VERSION,
        }

    @app.post("/predict",
              response_model=PredictionResponse,
              tags=["Predictions"],
              dependencies=[Depends(verify_api_key)])
    def predict(
        request: PredictionRequest,
        background_tasks: BackgroundTasks
    ) -> PredictionResponse:
        """Single prediction with API key authentication."""
        t0 = time.time()

        features = np.array(request.features).reshape(1, -1)
        features_scaled = scaler.transform(features)

        prediction = int(model.predict(features_scaled)[0])
        proba = model.predict_proba(features_scaled)[0].tolist()
        confidence = float(max(proba))
        latency_ms = (time.time() - t0) * 1000

        response = PredictionResponse(
            prediction=prediction,
            confidence=confidence,
            probabilities=proba if request.return_probabilities else None,
            latency_ms=round(latency_ms, 3),
            model_version=MODEL_VERSION,
        )

        # Log asynchronously — doesn't add to response latency
        background_tasks.add_task(
            log_prediction,
            request_data=request.model_dump(),
            response_data=response.model_dump()
        )
        return response

    @app.post("/predict/batch",
              response_model=BatchPredictionResponse,
              tags=["Predictions"],
              dependencies=[Depends(verify_api_key)])
    def predict_batch(request: BatchPredictionRequest) -> BatchPredictionResponse:
        """Batch predictions — more efficient than calling /predict N times."""
        t0 = time.time()
        X_batch = np.array(request.instances)
        X_scaled = scaler.transform(X_batch)
        predictions = model.predict(X_scaled).tolist()
        probabilities = model.predict_proba(X_scaled).tolist()
        return BatchPredictionResponse(
            predictions=predictions,
            probabilities=probabilities,
            n_instances=len(predictions),
            latency_ms=round((time.time() - t0) * 1000, 3),
        )

    @app.get("/predictions/log", tags=["Monitoring"])
    def get_prediction_log(limit: int = 10):
        """Return recent prediction log (for debugging)."""
        return {"n_predictions": len(prediction_log),
                "recent": prediction_log[-limit:]}

    print("FastAPI app created with endpoints:")
    for route in app.routes:
        if hasattr(route, 'methods'):
            methods = ','.join(route.methods)
            print(f"  {methods:8s} {route.path}")
else:
    print("[Simulated] FastAPI app with endpoints:")
    for m, p in [("GET", "/health"), ("GET", "/model-info"),
                 ("POST", "/predict"), ("POST", "/predict/batch"),
                 ("GET", "/predictions/log")]:
        print(f"  {m:8s} {p}")

## Core Concept 3: Testing with TestClient (No Server Needed!)

In [ ]:
if FASTAPI_AVAILABLE:
    client = TestClient(app)

    # ── Test 1: Health check ──────────────────────────────────────────────────
    print("Test 1: Health check")
    resp = client.get("/health")
    print(f"  Status: {resp.status_code}")
    print(f"  Body: {json.dumps(resp.json(), indent=4)}")
    print()

    # ── Test 2: Prediction with valid data ────────────────────────────────────
    print("Test 2: Valid prediction")
    sample_features = X_test[0].tolist()
    resp = client.post(
        "/predict",
        json={"features": sample_features, "return_probabilities": True},
        headers={"x-api-key": API_KEY}
    )
    print(f"  Status: {resp.status_code}")
    print(f"  Body: {json.dumps(resp.json(), indent=4)}")
    print()

    # ── Test 3: Missing API key ───────────────────────────────────────────────
    print("Test 3: Missing API key → 401")
    resp = client.post("/predict", json={"features": sample_features})
    print(f"  Status: {resp.status_code}")
    print(f"  Body: {resp.json()}")
    print()

    # ── Test 4: Invalid input (wrong number of features) ──────────────────────
    print("Test 4: Invalid input (5 features instead of 10) → 422")
    resp = client.post(
        "/predict",
        json={"features": [1.0, 2.0, 3.0, 4.0, 5.0]},  # only 5!
        headers={"x-api-key": API_KEY}
    )
    print(f"  Status: {resp.status_code} (422 = Unprocessable Entity)")
    print(f"  Error: {resp.json()['detail'][0]['msg']}")
    print()

    # ── Test 5: Batch prediction ──────────────────────────────────────────────
    print("Test 5: Batch prediction (5 samples)")
    batch = X_test[:5].tolist()
    resp = client.post(
        "/predict/batch",
        json={"instances": batch},
        headers={"x-api-key": API_KEY}
    )
    result = resp.json()
    print(f"  Status: {resp.status_code}")
    print(f"  Predictions: {result['predictions']}")
    print(f"  Latency: {result['latency_ms']} ms")

else:
    print("Simulated TestClient output:")
    print()
    print("Test 1 — GET /health → 200")
    print('  {"status": "healthy", "model_loaded": true, "uptime_seconds": 0.12}')
    print()
    print("Test 2 — POST /predict (valid) → 200")
    print('  {"prediction": 1, "confidence": 0.86, "probabilities": [0.14, 0.86], "latency_ms": 2.3}')
    print()
    print("Test 3 — POST /predict (no API key) → 401")
    print('  {"detail": "Invalid or missing API key"}')
    print()
    print("Test 4 — POST /predict (5 features) → 422")
    print('  {"detail": [{"msg": "List should have at least 10 items"}]}')
    print()
    print("Test 5 — POST /predict/batch (5 samples) → 200")
    print('  {"predictions": [1, 0, 1, 1, 0], "latency_ms": 3.7}')

## Core Concept 4: Deploying in Production

```bash
# Run locally
uvicorn main:app --host 0.0.0.0 --port 8000 --reload

# Production (multiple workers)
uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4

# With Gunicorn (process manager) + Uvicorn workers
gunicorn main:app -w 4 -k uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000

# Docker
FROM python:3.11-slim
COPY . .
RUN pip install fastapi uvicorn scikit-learn
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Loading model on every request | Extreme latency | Load model at startup using `lifespan` event handler |
| Blocking async functions | Performance degraded | Use `async def` for I/O, `def` for CPU |
| Not using Pydantic validation | Manual `if` checks everywhere | Define request models — FastAPI validates automatically |
| Returning numpy types | JSON serialization error | Convert: `.tolist()`, `int()`, `float()` |
| No input limits | OOM attack (batch of 1M samples) | Add `max_length` to list fields |
| Secrets in code | Security breach | Use environment variables: `os.environ['API_KEY']` |
| CORS not configured | Browser requests blocked | Add `CORSMiddleware` with appropriate origins |

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "FastAPI vs Flask for ML model serving — which would you choose and why?",
     "a": """FastAPI for new projects, Flask for legacy compatibility.

FastAPI advantages:
1. Speed: ~3× faster than Flask (ASGI vs WSGI, async support)
2. Automatic validation: Pydantic models = no manual input checking
3. Auto-generated docs: /docs and /redoc out of the box
4. Type hints: better IDE support, catch bugs before runtime
5. Async: can handle many requests simultaneously (good for I/O-bound work)
6. Modern Python: designed for Python 3.6+ with type annotations

Flask advantages:
1. More mature ecosystem, more tutorials
2. Simpler for very small APIs
3. Existing codebase may be Flask already

For ML serving specifically: FastAPI is almost always better.
The automatic Pydantic validation is particularly valuable — ML APIs receive
untrusted data from clients and must validate every input."""},

    {"q": "How would you serve an ML model efficiently at high throughput?",
     "a": """Several levels of optimization:

1. Load model once at startup (not per request):
   Use FastAPI 'lifespan' context manager:
   @asynccontextmanager
   async def lifespan(app):
       app.state.model = load_model()  # runs at startup
       yield
       # cleanup at shutdown

2. Use async endpoints for I/O-bound operations:
   async def predict(): → non-blocking, handles 1000s concurrent
   sync def predict():  → blocking, use for CPU-heavy models

3. Run multiple workers:
   uvicorn --workers 4  → 4 processes, each with own model copy

4. Enable batching:
   Accept lists of inputs, process as numpy arrays (vectorized)

5. Caching:
   Cache frequent identical requests with Redis or functools.lru_cache

6. Model optimization:
   ONNX export, quantization, TorchScript for faster inference

7. Load balancer:
   Nginx/K8s in front of multiple FastAPI instances"""},

    {"q": "What is Pydantic and why is it important for ML APIs?",
     "a": """Pydantic is a data validation library that uses Python type annotations.

For ML APIs, it's critical because:

1. Automatic validation:
   class Request(BaseModel):
       features: List[float] = Field(..., min_length=10, max_length=10)
   # FastAPI auto-validates; invalid → HTTP 422 with clear error message

2. Type coercion:
   '1.5' (string from JSON) → 1.5 (float) automatically

3. Documentation: Pydantic models → OpenAPI schema → /docs UI

4. Serialization: response_model= ensures output is correctly formatted

5. Custom validators: @field_validator for ML-specific rules:
   - Features must be finite (no NaN/Inf)
   - Batch size ≤ 1000
   - Image dimensions must be 224×224

Without Pydantic: you'd write 50 lines of manual validation code
for every endpoint. With Pydantic: 5 lines of schema definition."""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Feature | FastAPI API |
|---------|------------|
| Create app | `app = FastAPI(title=..., version=...)` |
| Define endpoint | `@app.get('/path')` / `@app.post('/path')` |
| Validate input | Pydantic `BaseModel` for request body |
| Validate output | `response_model=MyResponseModel` |
| Authentication | `Depends(verify_api_key)` dependency |
| CORS | `CORSMiddleware` |
| Background tasks | `BackgroundTasks.add_task(fn, args)` |
| Testing | `TestClient(app).get('/path')` |
| Run | `uvicorn main:app --host 0.0.0.0 --port 8000` |

### Next Steps
1. **Official tutorial**: [https://fastapi.tiangolo.com/tutorial/](https://fastapi.tiangolo.com/tutorial/)
2. **FastAPI + ML**: [https://fastapi.tiangolo.com/deployment/](https://fastapi.tiangolo.com/deployment/)
3. **Next**: Learn BentoML for ML-specific packaging and Ray Serve for distributed serving